# Course Correct Labs Reasoning Stability Observatory

**Unified Analysis System for CCL Empirical Studies**

This notebook provides a comprehensive analysis of four CCL studies:
1. **Mirror Loop** - Information collapse in iterative self-critique
2. **Recursive Confabulation** - Fabrication persistence across turns
3. **Violation State** - Contamination from refusal states
4. **Echo Chamber** - Belief percolation in multi-agent systems

**Runtime:** <10 minutes using existing data only

**Budget:** $0 (uses precomputed data; optional live demo <$0.05)

---

**To get started:** Just click **Runtime → Run all**

*For local use: `pip install -e .` from repo root, then run all cells*

## Setup & Installation

In [ ]:
# Setup: ensure we use the REPO version of course_correct_evals, not a stale pip wheel

import sys, os, subprocess

REPO_URL = "https://github.com/Course-Correct-Labs/course-correct-evals.git"
BRANCH = "claude/build-ccl-observatory-01Tm8d1ASgVx3NTHTqaXPEpt"
REPO_DIR = "/content/course-correct-evals"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🔧 Detected Google Colab environment")

    # 1) Uninstall any pip-installed version so it stops polluting imports
    try:
        print("🧹 Uninstalling pip package course-correct-evals (if present)...")
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "course-correct-evals"],
            check=False,
        )
    except Exception as e:
        print(f"⚠️ pip uninstall failed (ignoring): {e}")

    # 2) Clear any already-imported modules from this session
    for name in list(sys.modules.keys()):
        if name.startswith("course_correct_evals"):
            del sys.modules[name]
    print("🧽 Cleared course_correct_evals from sys.modules")

    # 3) Clone or update the repo
    if not os.path.exists(REPO_DIR):
        print(f"📦 Cloning repo from {REPO_URL} (branch {BRANCH})...")
        subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR],
            check=True,
        )
        print("✅ Repo cloned")
    else:
        print("✅ Repo already cloned, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

    # 4) Ensure repo is at the front of sys.path
    if REPO_DIR in sys.path:
        sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Added {REPO_DIR} to sys.path first")

else:
    # Local usage: assume user already has repo and editable install
    print("🏠 Not in Colab.")
    print("   For local use, run from inside the cloned repo or `pip install -e .`")

print("\n📚 Loading libraries...")

import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from course_correct_evals import (
    MirrorLoopImporter,
    ConfabulationImporter,
    ViolationStateImporter,
    EchoChamberImporter,
    CrossStudyAnalysis,
)

from course_correct_evals.analysis.viz import (
    plot_four_panel_comparison,
    plot_leaderboard,
    plot_mirror_loop_detail,
)

from course_correct_evals.reports import export_csv_results, export_pdf_report

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

# Plot settings
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")

warnings.filterwarnings("ignore")

import course_correct_evals, inspect
print("\n✅ Setup complete – ready to run!")
print("📍 course_correct_evals loaded from:", inspect.getfile(course_correct_evals))

## 1. Load All Studies

The Observatory will attempt to load data from all four studies.
Studies without available data will be skipped gracefully.

In [ ]:
# Initialize Observatory
observatory = CrossStudyAnalysis()

# Load all available studies
loaded_studies = observatory.load_all_studies(
    fail_on_missing=False  # Continue even if some studies missing
)

print("\nLoading Summary:")
data_sources = observatory.get_data_source_summary()
for study_name, info in data_sources.items():
    status_icon = "[LOADED]" if info['loaded'] else "[UNAVAILABLE]"
    source = info['source']
    
    # Format source for readability
    if source.startswith('remote:'):
        source_desc = "GitHub (remote)"
    elif source.startswith('local:'):
        source_desc = f"Local file"
    elif source.startswith('explicit_path:'):
        source_desc = f"Explicit path"
    elif source.startswith('env:'):
        source_desc = f"Environment variable"
    else:
        source_desc = "Not loaded"
    
    print(f"  {status_icon} {study_name:20s} - {source_desc}")

## 2. Cross-Study Leaderboard

Compare model performance across all four studies.

In [ ]:
# Generate leaderboard
leaderboard = observatory.create_leaderboard()

print("\nCross-Study Model Leaderboard:")
print("=" * 80)
display(leaderboard)

# Visualize leaderboard
if len(leaderboard) > 0:
    plot_leaderboard(leaderboard)

### Leaderboard Metrics Explanation

- **mirror_collapse_rate**: % of sequences that exhibit information collapse (lower is better)
- **confab_persistence_rate**: % of fabrications that persist after intervention (lower is better)
- **violation_contamination_rate**: % of turns contaminated by violation state (lower is better)
- **echo_mean_GR**: Mean Group Radicalization score (lower is better)
- **echo_mean_SRI**: Mean Self-Reinforcement Index (lower is better)

## 3. Four-Panel Comparison Figure

Publication-quality visualization comparing all four studies.

In [ ]:
# Generate flagship figure
fig = plot_four_panel_comparison(
    observatory,
    figsize=(16, 12),
    save_path='four_panel_comparison.png'
)

plt.show()

## 4. Mirror Loop Deep Dive

Detailed analysis of information collapse in iterative self-critique.

In [ ]:
if observatory._data_loaded['mirror_loop']:
    ml_analysis = observatory.analyze_mirror_loop()
    
    print("\nMirror Loop Study Results:")
    print("=" * 80)
    print(f"Total Sequences: {ml_analysis['total_sequences']}")
    print(f"Collapsed Sequences: {ml_analysis['collapsed_sequences']}")
    print(f"Collapse Rate: {ml_analysis['collapse_rate']:.1%}")
    print(f"Mean ΔI: {ml_analysis['mean_delta_i_overall']:.3f}")
    
    if ml_analysis['model_statistics'] is not None:
        print("\nPer-Model Statistics:")
        display(ml_analysis['model_statistics'])
    
    print("\nSample Sequence Analysis:")
    display(ml_analysis['sequence_analysis'].head(10))
    
    sequence_ids = observatory.mirror_loop_data['sequence_id'].unique()
    if len(sequence_ids) > 0:
        plot_mirror_loop_detail(observatory, sequence_id=sequence_ids[0])
        plt.show()
else:
    print("Mirror Loop data not available")

## 5. Recursive Confabulation Deep Dive

Analysis of fabrication persistence and intervention effectiveness.

In [ ]:
if observatory._data_loaded['confabulation']:
    conf_analysis = observatory.analyze_confabulation()
    
    print("\nRecursive Confabulation Study Results:")
    print("=" * 80)
    print(f"Total Conversations: {conf_analysis['total_conversations']}")
    print(f"Total Turns: {conf_analysis['total_turns']}")
    
    pers_overall = conf_analysis['persistence_statistics']['overall']
    print("\nOverall Persistence:")
    print(f"  Total Fabrications: {pers_overall['total_fabrications']}")
    print(f"  Persistent: {pers_overall['persistent_fabrications']}")
    print(f"  Persistence Rate: {pers_overall['persistence_rate']:.1%}")
    
    if 'by_intervention' in conf_analysis['persistence_statistics']:
        print("\nPersistence by Intervention Arm:")
        for arm, stats in conf_analysis['persistence_statistics']['by_intervention'].items():
            print(f"  {arm}:")
            print(f"    Fabrications: {stats['total_fabrications']}")
            print(f"    Persistence Rate: {stats['persistence_rate']:.1%}")
    
    if conf_analysis['intervention_effectiveness'] is not None:
        print("\nIntervention Effectiveness:")
        display(conf_analysis['intervention_effectiveness'])
else:
    print("Confabulation data not available")

## 6. Violation State Deep Dive

Analysis of contamination from violation requests and refusals.

In [ ]:
if observatory._data_loaded['violation_state']:
    vs_analysis = observatory.analyze_violation_state()
    
    print("\nViolation State Study Results:")
    print("=" * 80)
    print(f"Total Conversations: {vs_analysis['total_conversations']}")
    print(f"Total Turns: {vs_analysis['total_turns']}")
    
    contam_stats = vs_analysis['contamination_statistics']
    print("\nContamination Statistics:")
    print(f"  Contaminated Conversations: {contam_stats['contaminated_conversations']}")
    print(f"  Contaminated Turns: {contam_stats['contaminated_turns']}")
    print(f"  Contamination Rate: {contam_stats['contamination_rate']:.1%}")
    print(f"  Conversation Contamination Rate: {contam_stats['conversation_contamination_rate']:.1%}")
    
    print("\nRefusal Statistics:")
    display(vs_analysis['refusal_statistics'])
    
    if 'response_type' in observatory.violation_state_data.columns:
        print("\nResponse Type Distribution:")
        response_dist = observatory.violation_state_data['response_type'].value_counts()
        display(response_dist)
else:
    print("Violation State data not available")

## 7. Echo Chamber Deep Dive

Analysis of belief percolation and radicalization in multi-agent systems.

In [ ]:
if observatory._data_loaded['echo_chamber']:
    echo_analysis = observatory.analyze_echo_chamber()
    
    print("\nEcho Chamber Study Results:")
    print("=" * 80)
    print(f"Total Simulations: {echo_analysis['total_simulations']}")
    print(f"Total Steps: {echo_analysis['total_steps']}")
    
    echo_stats = echo_analysis['echo_statistics']
    print("\nEcho Chamber Metrics:")
    for metric_name, metric_stats in echo_stats['metrics'].items():
        print(f"\n{metric_name}:")
        print(f"  Mean: {metric_stats['mean']:.3f}")
        print(f"  Std: {metric_stats['std']:.3f}")
        print(f"  Range: [{metric_stats['min']:.3f}, {metric_stats['max']:.3f}]")
        if 'trend_mean' in metric_stats:
            print(f"  Mean Trend: {metric_stats['trend_mean']:.3f}")
            print(f"  Increasing Trajectories: {metric_stats['increasing_count']}")
            print(f"  Decreasing Trajectories: {metric_stats['decreasing_count']}")
    
    print("\nConvergence Statistics:")
    conv_stats = echo_analysis['convergence_statistics']
    for key, value in conv_stats.items():
        print(f"  {key}: {value}")
    
    print("\nSample Trajectories:")
    display(echo_analysis['trajectories'].head(10))
    
    print("\nThreshold Crossings:")
    for metric, crossings in echo_analysis['threshold_crossings'].items():
        if len(crossings) > 0:
            print(f"\n{metric}:")
            display(crossings.head())
else:
    print("Echo Chamber data not available")

## 8. Optional: Live Demo

**⚠️ WARNING: This cell costs money and requires API keys!**

To enable:
1. Set `RUN_LIVE_DEMO = True` below
2. Provide your API key
3. Run the cell

Estimated cost: $0.01-0.05 for 10 iterations

In [ ]:
# DISABLED BY DEFAULT
RUN_LIVE_DEMO = False

if RUN_LIVE_DEMO:
    print("WARNING: Running live demo - this will cost money!")
    
    from course_correct_evals.runners.mirror_loop_runner import (
        run_mirror_loop_demo,
        analyze_live_demo,
    )
    
    import course_correct_evals.runners.mirror_loop_runner as runner_module
    runner_module.RUN_LIVE_DEMO = True
    
    demo_results = run_mirror_loop_demo(
        prompt="Explain the concept of recursion in programming.",
        model="gpt-3.5-turbo",
        max_iterations=10,
        api_key=None
    )
    
    demo_analysis = analyze_live_demo(demo_results)
    
    print("\nLive Demo Results:")
    print("=" * 80)
    print(f"Sequence Length: {demo_analysis['length']}")
    print(f"Collapse Detected: {demo_analysis['collapse_detected']}")
    if demo_analysis['collapse_detected']:
        print(f"Collapse at Iteration: {demo_analysis['collapse_iteration']}")
    print(f"Mean ΔI: {demo_analysis['delta_i_edit_mean']:.3f}")
    print(f"Mean N-gram Novelty: {demo_analysis['ngram_novelty_mean']:.3f}")
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(demo_analysis['delta_i_edit']) + 1),
             demo_analysis['delta_i_edit'],
             'o-', linewidth=2, markersize=6)
    plt.axhline(y=demo_analysis['collapse_threshold'],
                color='red', linestyle='--', alpha=0.5,
                label=f"Threshold ({demo_analysis['collapse_threshold']})")
    plt.xlabel('Iteration')
    plt.ylabel('ΔI (Edit Distance)')
    plt.title('Live Demo: Information Change Over Iterations')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Live demo disabled. Set RUN_LIVE_DEMO = True to enable.")
    print("Note: This will cost money and require API keys.")

## 9. Export Results

Export all analysis results to CSV and generate a report.

In [ ]:
# Export CSV results
print("Exporting results to CSV...\n")
exported_files = export_csv_results(observatory, output_dir='results')

print("\nExported Files:")
for result_type, file_path in exported_files.items():
    print(f"  {result_type}: {file_path}")

In [ ]:
# Generate report
print("Generating report...\n")
report_path = export_pdf_report(
    observatory,
    output_path='ccl_observatory_report.pdf'
)

print(f"\nReport generated: {report_path}")

## Summary

This notebook has provided a comprehensive analysis of the CCL Reasoning Stability Observatory.

### Key Findings

The Observatory synthesizes data across four empirical studies to identify:

1. **Information Collapse** - Models that exhibit rapid ΔI decay in self-critique
2. **Fabrication Persistence** - Models prone to maintaining false information
3. **State Contamination** - Models whose refusal behavior leaks across contexts
4. **Echo Chamber Effects** - Multi-agent systems prone to radicalization

### Next Steps

1. Review exported CSV results in `results/` folder (or download from Colab)
2. Examine the four-panel comparison figure
3. Use the leaderboard to compare model stability
4. Optionally: Run live demos with your own prompts

---

**Course Correct Labs**  
*Reasoning Stability Observatory v0.1.0*